# Fat tails in the Nifty 50

Companion notebook to **The Indic Quant**, video 01. Every number in the video comes from a cell below.

**Education only, not investment advice.** Historical data, at least 3 months old.

**Data.** Nifty 50 daily closes from NSE Indices (niftyindices.com), 3 Nov 1995 (the index base date) to
30 Jun 2026. Download the CSV yourself and save it as `nifty50.csv` next to this notebook, or run as-is and the
notebook falls back to Yahoo Finance, which only starts in September 2007 (fat tails and volatility clustering show up
there too, but the daily-to-monthly thinning in Ch 6 needs the longer history). S&P 500 daily closes from Yahoo Finance, 1950 to 2026. See `nifty_data.py`.

In [1]:
import json
import platform

import numpy as np
import pandas as pd
from arch import arch_model
from scipy import stats

from nifty_data import fit_two_bells, load_nifty_closes, load_spx_closes

print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

python 3.12.10 | numpy 1.26.4 | pandas 2.2.1


## Ch 2 · Putting Nifty on the ruler

In [2]:
px = load_nifty_closes(start="1995-11-03", end="2026-06-30")
print(f"{len(px):,} closes, {px.index[0]:%d %b %Y} to {px.index[-1]:%d %b %Y}")

Nifty 50 from nifty50.csv
7,624 closes, 03 Nov 1995 to 30 Jun 2026


In [3]:
r = np.log(px).diff().dropna()        # daily log returns
mu, sigma = r.mean(), r.std()
z = (r - mu) / sigma                  # every day on the σ ruler

print(f"average day {mu:+.3%}   σ {sigma:.2%} a day")

average day +0.042%   σ 1.42% a day


In [4]:
for k in [3, 4, 5, 6]:
    seen = (z.abs() > k).sum()
    expected = len(z) * 2 * stats.norm.sf(k)   # two-sided
    print(f"beyond {k}σ   Nifty {seen:4d}   bell curve {expected:9.4f}")

beyond 3σ   Nifty  116   bell curve   20.5805
beyond 4σ   Nifty   45   bell curve    0.4829
beyond 5σ   Nifty   18   bell curve    0.0044
beyond 6σ   Nifty    8   bell curve    0.0000


In [5]:
print(f"skewness        {stats.skew(r):+.2f}")
print(f"excess kurtosis {stats.kurtosis(r):5.1f}   (bell curve: 0)")
print(f"average z⁴      {np.mean(z**4):5.1f}   (bell curve: 3)")

skewness        -0.32
excess kurtosis   9.4   (bell curve: 0)
average z⁴       12.4   (bell curve: 3)


In [6]:
for d in z.abs().nlargest(5).index:
    years = 1 / (2 * stats.norm.sf(abs(z[d]))) / 252
    print(f"{d:%d %b %Y}  {np.expm1(r[d]):+7.2%}  {z[d]:+5.1f}σ   bell curve: once in {years:.1e} years")

18 May 2009  +17.74%  +11.5σ   bell curve: once in 2.6e+27 years
23 Mar 2020  -12.98%   -9.8σ   bell curve: once in 4.8e+19 years
17 May 2004  -12.24%   -9.2σ   bell curve: once in 1.5e+17 years
24 Oct 2008  -12.20%   -9.2σ   bell curve: once in 1.2e+17 years
01 Mar 1997  +10.44%   +7.0σ   bell curve: once in 1.3e+09 years


## Ch 3 · The ruler keeps changing size

In [7]:
print(f"direction, today vs tomorrow: {r.autocorr(1):+.3f}")
print(f"size,      today vs tomorrow: {r.abs().autocorr(1):+.3f}")

vol = r.rolling(21).std() * np.sqrt(252)    # one-month volatility, annualised
print(f"one-month volatility ran from {vol.min():.0%} to {vol.max():.0%}")

direction, today vs tomorrow: +0.047
size,      today vs tomorrow: +0.270
one-month volatility ran from 5% to 89%


In [8]:
scaled = r / r.rolling(21).std().shift(1)   # each day against the month before it
print(f"excess kurtosis  raw {stats.kurtosis(r):.1f}  ->  rescaled {stats.kurtosis(scaled.dropna()):.1f}")

excess kurtosis  raw 9.4  ->  rescaled 2.7


## Ch 4 · Mixing bell curves

In [9]:
w, s_calm, s_storm = fit_two_bells(r)
print(f"calm bell   {w:.0%} of days, σ {s_calm:.2%}")
print(f"storm bell  {1 - w:.0%} of days, σ {s_storm:.2%}")

sd = np.sqrt(w * s_calm**2 + (1 - w) * s_storm**2)
p_mix = 2 * (w * stats.norm.sf(5 * sd / s_calm) + (1 - w) * stats.norm.sf(5 * sd / s_storm))
print(f"P(beyond 5σ)  one bell {2 * stats.norm.sf(5):.1e}   two bells {p_mix:.1e}")

calm bell   82% of days, σ 0.92%
storm bell  18% of days, σ 2.69%
P(beyond 5σ)  one bell 5.7e-07   two bells 1.5e-03


## Ch 5 · The days that are still impossible

In [10]:
fit = arch_model(100 * r, vol="GARCH", p=1, q=1, dist="t").fit(disp="off")
cz = fit.std_resid                     # each day against its own GARCH σ

days = ["2020-03-23", "2004-05-17", "2009-05-18", "2024-06-04"]
for d in [d for d in days if d in z.index]:
    print(f"{d}   fixed ruler {z[d]:+5.1f}σ   its own σ {cz[d]:+5.1f}σ")

2020-03-23   fixed ruler  -9.8σ   its own σ  -3.4σ
2004-05-17   fixed ruler  -9.2σ   its own σ  -4.3σ
2009-05-18   fixed ruler +11.5σ   its own σ  +7.5σ
2024-06-04   fixed ruler  -4.3σ   its own σ  -5.1σ


## Ch 6 · Zoom out: days, weeks, months

In [11]:
for name, rule in [("daily", None), ("weekly", "W-FRI"), ("monthly", "ME")]:
    x = r if rule is None else r.resample(rule).sum(min_count=1).dropna()
    print(f"{name:8s} {len(x):5,} points   excess kurtosis {stats.kurtosis(x):.1f}")

daily    7,623 points   excess kurtosis 9.4
weekly   1,600 points   excess kurtosis 3.2
monthly    368 points   excess kurtosis 2.4


## Ch 7 · Nifty vs S&P 500

In [12]:
def hill(x, k):
    x = np.sort(x[x > 0])[::-1]
    return 1 / np.mean(np.log(x[:k] / x[k]))

for k in [50, 100, 200, 400]:
    print(f"k={k:3d}   tail exponent  losses {hill(-r.values, k):.2f}   gains {hill(r.values, k):.2f}")

k= 50   tail exponent  losses 3.28   gains 3.43
k=100   tail exponent  losses 3.10   gains 3.36
k=200   tail exponent  losses 2.72   gains 3.08
k=400   tail exponent  losses 2.65   gains 2.83


In [13]:
spx = load_spx_closes(start="1950-01-01", end="2026-06-30")
r_spx = np.log(spx).diff().dropna()

for name, x in [("Nifty 50", r), ("S&P 500", r_spx)]:
    d4 = (x - x.mean()) ** 4
    big = d4.idxmax()
    print(f"{name}: excess kurtosis {stats.kurtosis(x):4.1f}; {big:%d %b %Y} alone is {d4[big] / d4.sum():.0%} of the z⁴ sum")

Nifty 50: excess kurtosis  9.4; 18 May 2009 alone is 18% of the z⁴ sum
S&P 500: excess kurtosis 25.2; 19 Oct 1987 alone is 52% of the z⁴ sum


## Ch 8 · What fat tails break

In [14]:
for q in [0.99, 0.999]:
    var = -(mu + sigma * stats.norm.ppf(1 - q))       # bell-curve VaR, as a log return
    hits = (r < -var).sum()
    expected = len(r) * (1 - q)
    print(f"{q:.1%} VaR {-np.expm1(-var):.2%}: breached {hits} times, bell curve expects {expected:.0f}")

99.0% VaR 3.21%: breached 144 times, bell curve expects 76
99.9% VaR 4.25%: breached 58 times, bell curve expects 8


The headline numbers in one place, saved to `results.json`.

In [15]:
results = {
    "n_days": len(r), "sd_daily_pct": round(sigma * 100, 2),
    "skew": round(stats.skew(r), 2), "excess_kurtosis": round(stats.kurtosis(r), 1),
    "beyond": {k: int((z.abs() > k).sum()) for k in [3, 4, 5, 6]},
    "kurt_trailing_month": round(stats.kurtosis(scaled.dropna()), 1),
    "mixture": {"w_calm": round(w, 3), "sd_calm_pct": round(s_calm * 100, 2), "sd_storm_pct": round(s_storm * 100, 2)},
    "z_cond": {d: round(float(cz[d]), 1) for d in days if d in cz.index},
    "aggregation_kurt": {n: round(stats.kurtosis(r if q is None else r.resample(q).sum(min_count=1).dropna()), 1)
                         for n, q in [("daily", None), ("weekly", "W-FRI"), ("monthly", "ME")]},
    "var_breaches": {str(q): int((r < mu + sigma * stats.norm.ppf(1 - q)).sum()) for q in [0.99, 0.999]},
}
json.dump(results, open("results.json", "w"), indent=1)
results

{'n_days': 7623,
 'sd_daily_pct': 1.42,
 'skew': -0.32,
 'excess_kurtosis': 9.4,
 'beyond': {3: 116, 4: 45, 5: 18, 6: 8},
 'kurt_trailing_month': 2.7,
 'mixture': {'w_calm': 0.818, 'sd_calm_pct': 0.92, 'sd_storm_pct': 2.69},
 'z_cond': {'2020-03-23': -3.4,
  '2004-05-17': -4.3,
  '2009-05-18': 7.5,
  '2024-06-04': -5.1},
 'aggregation_kurt': {'daily': 9.4, 'weekly': 3.2, 'monthly': 2.4},
 'var_breaches': {'0.99': 144, '0.999': 58}}